<a href="https://colab.research.google.com/github/profsuccodifrutta/ai_act_RAG_navigator/blob/main/public_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# --- CELL 1: GITHUB REPOSITORY SETUP ---
import os

# Configuration for GitHub
GIT_USER = "profsuccodifrutta"
GIT_REPO = "ai_act_RAG_navigator"
REPO_URL = f"https://github.com/{GIT_USER}/{GIT_REPO}.git"

# Clone repo
if not os.path.exists(GIT_REPO):
    print(f"Cloning repository {GIT_REPO}...")
    !git clone {REPO_URL}
else:
    print(f"Folder {GIT_REPO} already exists.")

# Move into the repo directory
%cd {GIT_REPO}

Cloning repository ai_act_RAG_navigator...
Cloning into 'ai_act_RAG_navigator'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 151 (delta 89), reused 12 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 2.50 MiB | 23.29 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/ai_act_RAG_navigator


In [3]:
# --- CELL 2: SYSTEM SETUP & DEPENDENCIES ---
!curl -LsSf https://astral.sh/uv/install.sh | sh
import sys
import os

# Add uv to PATH
sys.path.append("/root/.cargo/bin")

# Initialize uv
if not os.path.exists("pyproject.toml"):
    !uv init
else:
    print("uv already initialized.")

# Add and install ALL required libraries (Now includes TF, sklearn, pandas, etc.)
print("Installing libraries... This may take a few minutes due to TensorFlow and PyTorch.")
!uv add langchain langchain-community langchain-google-genai faiss-cpu pymupdf spacy python-dotenv langchain-text-splitters langchain-core transformers datasets sentence-transformers rank_bm25 pandas matplotlib scikit-learn plotly scipy tensorflow
!uv pip install --system langchain langchain-community langchain-google-genai faiss-cpu pymupdf spacy python-dotenv langchain-text-splitters langchain-core transformers datasets sentence-transformers rank_bm25 pandas matplotlib scikit-learn plotly scipy tensorflow

print("\nEnvironment configuration completed!")

downloading uv 0.10.11 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv already initialized.
Installing libraries... This may take a few minutes due to TensorFlow and PyTorch.
Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 169 packages in 1.18s
Prepared 166 packages in 2m 54s
Installed 166 packages in 1.66s
 + absl-py==2.4.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + astunparse==1.6.3
 + attrs==25.4.0
 + blis==1.3.3
 + catalogue==2.0.10
 + certifi==2026.1.4
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + cloudpathlib==0.23.0
 + confection==0.1.5
 + contourpy==1.3.3
 + cryptography==46.0.5
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.4.3
 + cycler==0.12.1
 + cymem==2.0.13
 + dataclasses-json==0.6.7
 + datasets==4.8.2
 + dill==0.4.1
 + distro==1.9.0
 + faiss-c

In [4]:
# --- CELL 3: IMPORTS ---
from google.colab import drive
import os
import requests
import shutil
import fitz  # This is PyMuPDF
import re
import pandas as pd
import matplotlib.pyplot as plt
import gc
import random
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
from spacy import displacy
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import plotly.graph_objects as go
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import plotly.io as pio
import plotly.express as px
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from scipy.special import softmax
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from collections import defaultdict
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, TextVectorization, Input, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import PolynomialDecay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

print("All libraries successfully imported!")

All libraries successfully imported!


In [5]:
# --- CELL 3: DATA & RAG CONFIGURATION ---
import os

# Target the PDF directly in the repository root
local_pdf_path = "ai_act_2024.pdf"

if os.path.exists(local_pdf_path):
    print(f"PDF successfully found locally at: {local_pdf_path}")
else:
    print(f"Error: The file '{local_pdf_path}' was not found.")
    print("Please verify the exact name on GitHub and ensure you ran Cell 1.")

# Local persistence folder for FAISS index
local_persistence_path = "ai_act_rag_index"
if not os.path.exists(local_persistence_path):
    os.makedirs(local_persistence_path)
    print("Folder for index persistence ready locally.")
else:
    print("Folder for index persistence already exists.")

PDF successfully found locally at: ai_act_2024.pdf
Folder for index persistence ready locally.


In [6]:
def parse_ai_act_robust(pdf_path):
    doc = fitz.open(pdf_path)

    categories_text = {
        "RECITAL": [],
        "ARTICLE": [],
        "ANNEX": []
    }

    noise_pattern = re.compile(r"(\d+/144|ELI: http://data\.europa\.eu/eli/reg/.+|OJ L, 12\.7\.2024|EN|\(Text with EEA relevance\))")
    footnote_cleaner = re.compile(r"\n\s*\(\s*\d+\s*\)\s*(?:OJ\b|Position\b|Directive\b|Regulation\b|See\b|Case\b).*$", re.IGNORECASE | re.DOTALL)

    footnotes_removed = 0

    for page_num in range(len(doc)):
        text = doc[page_num].get_text("text").replace("-\n", "")

        clean_lines = []
        for line in text.split("\n"):
            cleaned_line = noise_pattern.sub("", line).strip()
            if cleaned_line and not cleaned_line.isdigit():
                clean_lines.append(cleaned_line)

        # Join the page back together
        final_page_text = "\n" + "\n".join(clean_lines)

        if footnote_cleaner.search(final_page_text):
            # Replace the footnote block with a single newline
            final_page_text = footnote_cleaner.sub("\n", final_page_text)
            footnotes_removed += 1

        p_idx = page_num + 1
        if p_idx < 44:
            categories_text["RECITAL"].append(final_page_text)
        elif 44 <= p_idx <= 123:
            categories_text["ARTICLE"].append(final_page_text)
        else:
            categories_text["ANNEX"].append(final_page_text)

    doc.close()
    print(f"Removed footnote blocks from {footnotes_removed} pages.")

    units = []

    # proces recitals
    recital_text = "\n" + "\n".join(categories_text["RECITAL"]) + "\n"
    rec_parts = re.split(r"\n\s*\(\s*(\d+)\s*\)\s+", recital_text)
    for i in range(1, len(rec_parts), 2):
        text_content = rec_parts[i+1].replace("\n", " ").strip()
        if text_content:
            units.append({"id": f"RECITAL {rec_parts[i]}", "category": "RECITAL", "text": text_content})

    # proces articles
    article_text = "\n" + "\n".join(categories_text["ARTICLE"]) + "\n"
    art_parts = re.split(r"(?i)\n\s*Article\s+(\d+)\s*\n", article_text)
    for i in range(1, len(art_parts), 2):
        text_content = art_parts[i+1].replace("\n", " ").strip()
        if text_content:
            units.append({"id": f"ARTICLE {art_parts[i]}", "category": "ARTICLE", "text": text_content})

    # proces annexes
    annex_text = "\n" + "\n".join(categories_text["ANNEX"]) + "\n"
    annex_parts = re.split(r"(?i)\n\s*ANNEX\s+([IVXLCDM]+)\s*\n", annex_text)
    for i in range(1, len(annex_parts), 2):
        text_content = annex_parts[i+1].replace("\n", " ").strip()
        if text_content:
            units.append({"id": f"ANNEX {annex_parts[i].upper()}", "category": "ANNEX", "text": text_content})

    return pd.DataFrame(units)

In [7]:
df_units = parse_ai_act_robust(local_pdf_path)
df_units['len'] = df_units['text'].str.len()

print(f"Clean Units created: {len(df_units)}")
print(f"Max length: {df_units['len'].max()}")

Removed footnote blocks from 23 pages.
Clean Units created: 307
Max length: 17095


In [8]:
avg_len = df_units['len'].mean()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_units['len'],
    nbinsx=50,
    marker_color='#e74c3c',
    marker_line=dict(color='white', width=1),
    name='Unit Lengths'
))

fig.add_vline(x=1500, line_dash="dash", line_color="#c0392b")

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='#c0392b', dash='dash'),
    name='Target RAG Limit (1500)'
))

fig.add_vline(x=avg_len, line_dash="solid", line_color="#f39c12")

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='#f39c12', shape='linear'),
    name=f'Average Length ({avg_len:.0f})'
))

fig.update_layout(
    title='<b>Distribution of Legal Unit Lengths </b>',
    xaxis_title="Characters",
    yaxis_title="Number of Units",
    template='plotly_white',
    bargap=0.05,
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255, 255, 255, 0.5)"
    ),
    height=500
)


fig.show()

In [9]:
nlp = spacy.load("en_core_web_sm")

ruler = nlp.add_pipe("entity_ruler", before="ner") # ner = spacy module for ner
                                                   # before the model prediction

patterns = [
    # organizations and authoroties
    {"label": "ORG", "pattern": "AI Office"},
    {"label": "ORG", "pattern": "European Artificial Intelligence Board"},
    {"label": "ORG", "pattern": "Market Surveillance Authority"},
    {"label": "ORG", "pattern": [{"lower": "european"}, {"lower": "commission"}]},
    {"label": "ORG", "pattern": "European Data Protection Board"},
    {"label": "ORG", "pattern": "European Data Protection Supervisor"},
    {"label": "ORG", "pattern": "EDPB"},
    {"label": "ORG", "pattern": "EDPS"},
    {"label": "ORG", "pattern": "Board"},
    {"label": "ORG", "pattern": "Commission"},

    # geopolitical entities
    {"label": "GPE", "pattern": "Union"},
    {"label": "GPE", "pattern": "EU"},

    # documents
    {"label": "DOCUMENT", "pattern": "Regulation"},
    {"label": "DOCUMENT", "pattern": "Directive"},
    {"label": "DOCUMENT", "pattern": [{"lower": "annex"}, {"IS_ALPHA": True, "IS_UPPER": True}]},
    {"label": "DOCUMENT", "pattern": [{"lower": "annexes"}, {"IS_ALPHA": True, "IS_UPPER": True}]},

    # Chapters
    {"label": "LAW", "pattern": [{"lower": "chapter"}, {"IS_DIGIT": True}]},
    {"label": "LAW", "pattern": [{"lower": "chapter"}, {"IS_ALPHA": True, "IS_UPPER": True}]},

    # Sections
    {"label": "LAW", "pattern": [{"lower": "section"}, {"IS_DIGIT": True}]},
    {"label": "LAW", "pattern": [{"lower": "section"}, {"IS_ALPHA": True, "IS_UPPER": True}]},
    {"label": "LAW", "pattern": [{"lower": "sections"}, {"IS_DIGIT": True}]},
    {"label": "LAW", "pattern": [{"lower": "sections"}, {"IS_ALPHA": True, "IS_UPPER": True}]},

    # Articles
    {"label": "LAW", "pattern": [{"lower": "article"}, {"IS_DIGIT": True}]},

    # ignore these patterns
    {"label": "IGNORE", "pattern": "AI"},
    {"label": "IGNORE", "pattern": [{"lower": "artificial"}, {"lower": "intelligence"}]},
    {"label": "IGNORE", "pattern": [{"lower": "ai"}, {"lower": "system"}]},

    # concepts and legal roles
    {"label": "CONCEPT", "pattern": "CE"},
    {"label": "CONCEPT", "pattern": [{"text": "CE"}, {"lower": "marking"}]},
    {"label": "CONCEPT", "pattern": [{"lower": "general"}, {"IS_PUNCT": True, "OP": "?"}, {"lower": "purpose"}, {"lower": "ai"}]},
    {"label": "LEGAL_ROLE", "pattern": [{"lower": "provider"}]},
    {"label": "LEGAL_ROLE", "pattern": [{"lower": "providers"}]},
    {"label": "LEGAL_ROLE", "pattern": [{"lower": "deployer"}]},
    {"label": "LEGAL_ROLE", "pattern": [{"lower": "deployers"}]},
    # risk levels
    {"label": "RISK_LEVEL", "pattern": [{"lower": "high"}, {"IS_PUNCT": True, "OP": "?"}, {"lower": "risk"}]},
    {"label": "RISK_LEVEL", "pattern": [{"lower": "unacceptable"}, {"lower": "risk"}]},
    {"label": "RISK_LEVEL", "pattern": [{"lower": "limited"}, {"lower": "risk"}]},
    {"label": "RISK_LEVEL", "pattern": [{"lower": "minimal"}, {"lower": "risk"}]},
]

ruler.add_patterns(patterns)

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

final_documents = []


for _, row in df_units.iterrows():
    unit_id = row['id']
    category = row['category']
    text = row['text']

    chunks = text_splitter.split_text(text)

    for i, chunk in enumerate(chunks):
        enriched_content = f"[{unit_id}] {chunk}"

        combined_metadata = {
            "source": "EU_AI_Act_2024",
            "category": category,
            "legal_unit": unit_id,
            "chunk_index": i,
            "is_continuation": i > 0
        }

        spacy_doc = nlp(enriched_content)
        extracted = defaultdict(set)

        for ent in spacy_doc.ents:
            if ent.label_ != "IGNORE":
                extracted[ent.label_].add(ent.text)

        for label, entity_set in extracted.items():
            combined_metadata[f"ner_{label}"] = list(entity_set)

        doc = Document(
            page_content=enriched_content,
            metadata=combined_metadata
        )

        final_documents.append(doc)

print(f"Total vectorized chunks: {len(final_documents)}")

Total vectorized chunks: 577


In [11]:
sample_doc = random.choice(final_documents)

for key, value in sample_doc.metadata.items():
    if not key.startswith("ner_"):
        clean_key = key.replace('_', ' ').title()
        print(f"{clean_key}: {value}")


found_entities = False
for key, value in sample_doc.metadata.items():
    if key.startswith("ner_"):
        found_entities = True
        print(f"{key}: {value}")

if not found_entities:
    print("(No entities found in this specific chunk)")

print("\n--- Text Preview ---")
print(f"{sample_doc.page_content[:800]}...")

Source: EU_AI_Act_2024
Category: ANNEX
Legal Unit: ANNEX VIII
Chunk Index: 2
Is Continuation: True
ner_DOCUMENT: ['Directive', 'Regulation', 'ANNEX VIII']
ner_CARDINAL: ['5', '4', '3', '2016/680', '8', '7', '9', '6', '2016/679', '1', '2']
ner_LAW: ['Article 6(3)based', 'Section C', 'Article 35', 'Article 49(3', 'Article 6(3', 'Article 26(8', 'Article 27']
ner_RISK_LEVEL: ['high-risk']
ner_GPE: ['EU', 'Union']
ner_LEGAL_ROLE: ['deployer', 'deployers', 'provider']

--- Text Preview ---
[ANNEX VIII] . A description of the intended purpose of the AI system; 6. The condition or conditions under Article 6(3)based on which the AI system is considered to be not-high-risk; 7. A short summary of the grounds on which the AI system is considered to be not-high-risk in application of the procedure under Article 6(3); 8. The status of the AI system (on the market, or in service; no longer placed on the market/in service, recalled); 9. Any Member States in which the AI system has been placed on the m

In [12]:
polished_documents = []
deleted_chunks = []

for doc in final_documents:
    clean_text = re.sub(r"^(\[.*?\])\s*[,.]+\s*", r"\1 ", doc.page_content)

    if len(clean_text) > 100:
        doc.page_content = clean_text
        polished_documents.append(doc)
    else:
        doc.page_content = clean_text
        deleted_chunks.append(doc)

print(f"Original chunks: {len(final_documents)}")
print(f"Polished chunks ready for vectorization: {len(polished_documents)}")
print(f"Removed {len(deleted_chunks)} low-value chunks.\n")

Original chunks: 577
Polished chunks ready for vectorization: 570
Removed 7 low-value chunks.



In [13]:
import json
import os

save_path = "data"

if not os.path.exists(save_path):
    os.makedirs(save_path)

polished_data = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in polished_documents
]

file_path = os.path.join(save_path, "polished_chunks.json")

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(polished_data, f, ensure_ascii=False, indent=2)

print(f"Chunks successfully saved to {file_path}")

Chunks successfully saved to data/polished_chunks.json


In [14]:
print(" DELETED CHUNKS PREVIEW ")
if not deleted_chunks:
    print("No chunks were deleted.")
else:
    for i, doc in enumerate(deleted_chunks):
        print(f"\n--- Deleted Chunk {i+1} ---")
        print(f"Legal Unit: {doc.metadata.get('legal_unit', 'Unknown')}")
        print(f"Length: {len(doc.page_content)} characters")
        print(f"Text: '{doc.page_content}'")

 DELETED CHUNKS PREVIEW 

--- Deleted Chunk 1 ---
Legal Unit: RECITAL 8
Length: 12 characters
Text: '[RECITAL 8] '

--- Deleted Chunk 2 ---
Legal Unit: RECITAL 35
Length: 31 characters
Text: '[RECITAL 35] Such data includes'

--- Deleted Chunk 3 ---
Legal Unit: ARTICLE 3
Length: 12 characters
Text: '[ARTICLE 3] '

--- Deleted Chunk 4 ---
Legal Unit: ARTICLE 5
Length: 37 characters
Text: '[ARTICLE 5] Prohibited AI practices 1'

--- Deleted Chunk 5 ---
Legal Unit: ARTICLE 13
Length: 13 characters
Text: '[ARTICLE 13] '

--- Deleted Chunk 6 ---
Legal Unit: ARTICLE 16
Length: 13 characters
Text: '[ARTICLE 16] '

--- Deleted Chunk 7 ---
Legal Unit: ARTICLE 66
Length: 13 characters
Text: '[ARTICLE 66] '


In [15]:
sample_chunks = random.sample(polished_documents, 3)

for i, doc in enumerate(sample_chunks):
    print(f"\n--- CHUNK {i+1} (Source: {doc.metadata['legal_unit']}) ---")
    spacy_doc = nlp(doc.page_content)

    custom_colors = {
        "LEGAL_ROLE": "linear-gradient(90deg, #f1c40f, #f39c12)",
        "RISK_LEVEL": "linear-gradient(90deg, #e74c3c, #c0392b)",
        "CONCEPT": "linear-gradient(90deg, #3498db, #2980b9)",
        "DOCUMENT": "linear-gradient(90deg, #9b59b6, #8e44ad)",
        "LAW": "linear-gradient(90deg, #34495e, #2c3e50)",
        "GPE": "linear-gradient(90deg, #2ecc71, #27ae60)"
    }

    #  hides the 'IGNORE' tags
    visible_ents = ["ORG", "GPE", "DOCUMENT", "LAW", "CONCEPT", "LEGAL_ROLE", "RISK_LEVEL"]

    options = {
        "colors": custom_colors,
        "ents": visible_ents
    }

    displacy.render(spacy_doc, style="ent", jupyter=True, options=options)


--- CHUNK 1 (Source: ARTICLE 59) ---



--- CHUNK 2 (Source: ARTICLE 26) ---



--- CHUNK 3 (Source: RECITAL 172) ---


In [16]:
all_entity_counts = Counter()

for doc in polished_documents:
    for key, val_list in doc.metadata.items():
        if key.startswith("ner_"):
            clean_label = key.replace("ner_", "")
            all_entity_counts[clean_label] += len(val_list)

label_df = pd.DataFrame(all_entity_counts.most_common(), columns=['Entity Type', 'Count'])

fig = px.bar(
    label_df,
    x='Entity Type',
    y='Count',
    title='Named Entity Type Distribution (All Entities)',
    color='Count',
    color_continuous_scale='RdBu_r',
    text_auto=True
)

fig.update_layout(
    xaxis_title='Entity Type',
    yaxis_title='Frequency',
    xaxis_tickangle=-45,
    template='plotly_white',
    showlegend=False
)

fig.show()

In [17]:
chunk_lengths = [len(doc.page_content) for doc in polished_documents]
avg_len = sum(chunk_lengths) / len(chunk_lengths) if chunk_lengths else 0

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=chunk_lengths,
    nbinsx=50,
    marker_color='#e74c3c',
    marker_line=dict(color='white', width=1),
    name='Chunk Lengths'
))

fig.add_vline(x=1500, line_dash="dash", line_color="#c0392b")

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='#c0392b', dash='dash'),
    name='Max Chunk Limit (1500)'
))

fig.add_vline(x=avg_len, line_dash="solid", line_color="#f39c12")


fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='#f39c12'),
    name=f'Average Length ({avg_len:.0f})'
))

fig.update_layout(
    title='<b>Distribution of Final Vectorized Chunk Lengths</b>',
    xaxis_title="Characters per Chunk",
    yaxis_title="Number of Chunks",
    template='plotly_white',
    bargap=0.05,
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)"
    ),
    height=500
)


fig.show()

In [18]:
categories = [doc.metadata['category'] for doc in polished_documents]
category_counts = Counter(categories)
sorted_categories = sorted(category_counts.items(), key=lambda x: x[1], reverse=True)

labels = [x[0] for x in sorted_categories]
values = [x[1] for x in sorted_categories]

fig = go.Figure(data=[
    go.Bar(
        x=labels,
        y=values,
        text=values,
        textposition='auto',
        marker=dict(
            color=values,
            colorscale=[
                [0, '#ffcccc'],
                [1, '#b30000']
            ],
            showscale=False,
            line=dict(color='white', width=1.5)
        )
    )
])

fig.update_layout(
    title='<b>Document Structural Balance</b><br><sup>Chunks per Category</sup>',
    xaxis_title="Category",
    yaxis_title="Number of Vectorized Chunks",
    template='plotly_white',
    font=dict(family="Arial", size=12),
    bargap=0.2,
    height=500
)


fig.show()

In [19]:
corpus = [doc.page_content for doc in polished_documents]

custom_stop_words = ['shall', 'article', 'regulation', 'paragraph', 'annex',
                     'member', 'states', 'union', 'eu', 'referred', 'accordance', 'chapter', 'recital']

all_stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words

vectorizer = TfidfVectorizer(stop_words=all_stop_words, max_features=1000)
tfidf_matrix = vectorizer.fit_transform(corpus)

word_scores = tfidf_matrix.sum(axis=0).A1
words = vectorizer.get_feature_names_out()

df_words = pd.DataFrame({'word': words, 'score': word_scores})
top_20_words = df_words.sort_values(by='score', ascending=False).head(20)

top_20_words = top_20_words.sort_values(by='score', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=top_20_words['word'],
    x=top_20_words['score'],
    orientation='h',
    marker=dict(
        color='#9b59b6',
        line=dict(color='white', width=1)
    ),
    name='TF-IDF Score'
))

fig.update_layout(
    title='<b>Top 20 Most Important Terms in the AI Act (TF-IDF)</b>',
    xaxis_title="Cumulative TF-IDF Score",
    yaxis_title="Terms",
    template='plotly_white',
    height=700,
    yaxis={'categoryorder':'total ascending'},
    margin=dict(l=150)
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(0,0,0,0.1)')

fig.show()

In [20]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Model loaded")

print(f"Embedding {len(polished_documents)} chunks. This will take a moment...")

vectorstore = FAISS.from_documents(polished_documents, hf_embeddings) # text to gpu, vectorizes, and organizes in FAISS index
print("FAISS index built")

vectorstore.save_local(local_persistence_path)

print("\n Vector database created and saved")

/tmp/ipykernel_3563/174533229.py:1: LangChainDeprecationWarning:

The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded
Embedding 570 chunks. This will take a moment...
FAISS index built

 Vector database created and saved


In [21]:
# instanciating the model and loading database
hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)


vectorstore = FAISS.load_local(
    folder_path=local_persistence_path,
    embeddings=hf_embeddings,
    allow_dangerous_deserialization=True
)
print("Database loaded")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Database loaded


In [29]:
#  INITIALIZE RETRIEVAL ENGINE

# Find all NER keys
all_possible_ner_keys = set()
for doc in polished_documents:
    for key in doc.metadata.keys():
        if key.startswith("ner_"):
            all_possible_ner_keys.add(key)
all_possible_ner_keys = sorted(list(all_possible_ner_keys))

# Initialize the individual retrievers
bm25_retriever = BM25Retriever.from_documents(polished_documents)
bm25_retriever.k = 3
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Combine them into the hybrid engine
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5] # 50% keyword, 50% semantic
)

print("✅ Engine ready! (all_possible_ner_keys and hybrid_retriever are loaded into memory)")

✅ Engine ready! (all_possible_ner_keys and hybrid_retriever are loaded into memory)


In [30]:
query = "What are the registration obligations for high-risk biometric systems?"
print(f"User question: {query}\n")

bm25_retriever.k = 15
faiss_retriever.search_kwargs = {"k": 15}

raw_hybrid_chunks = hybrid_retriever.invoke(query)

filtered_chunks = []
for chunk in raw_hybrid_chunks:
    legal_roles = chunk.metadata.get("ner_LEGAL_ROLE", [])

    if any("deployer" in role.lower() for role in legal_roles):
        filtered_chunks.append(chunk)

if not filtered_chunks:
    print("No chunks found with 'deployer' metadata.")
else:
    for i, chunk in enumerate(filtered_chunks):
        print(f"FILTERED HYBRID RESULT {i+1}")
        print("-" * 60)

        # standard metadata
        print("GENERAL METADATA:")
        for key, value in chunk.metadata.items():
            if not key.startswith("ner_"):
                clean_key = key.replace('_', ' ').title()
                print(f"  • {clean_key}: {value}")

        # 2. Print NER metadata
        print("\nEXTRACTED ENTITIES:")
        has_entities = False

        for ner_key in all_possible_ner_keys:
            display_name = ner_key.replace('ner_', '').upper()
            extracted_entities = chunk.metadata.get(ner_key, [])

            if extracted_entities:
                if ner_key == "ner_LEGAL_ROLE":
                    print(f"  • {display_name} (TARGET FOUND): {extracted_entities}")
                else:
                    print(f"  • {display_name}: {extracted_entities}")
                has_entities = True

        if not has_entities:
            print("  • No entities detected.")

        print("\nTEXT PREVIEW:")
        print(f"{chunk.page_content[:500]}...\n")
        print("=" * 60 + "\n")

bm25_retriever.k = 3
faiss_retriever.search_kwargs = {"k": 3}

User question: What are the registration obligations for high-risk biometric systems?

FILTERED HYBRID RESULT 1
------------------------------------------------------------
GENERAL METADATA:
  • Source: EU_AI_Act_2024
  • Category: ARTICLE
  • Legal Unit: ARTICLE 26
  • Chunk Index: 3
  • Is Continuation: True

EXTRACTED ENTITIES:
  • CARDINAL: ['2016/680', '8.', '9', '10', '2016/679']
  • DOCUMENT: ['Directive', 'Regulation']
  • GPE: ['EU', 'Union']
  • LAW: ['Article 13', 'Article 49', 'Article 35', 'Article 71', 'ARTICLE 26', 'Article 27']
  • LEGAL_ROLE (TARGET FOUND): ['Deployers', 'deployer', 'deployers', 'provider']
  • RISK_LEVEL: ['high-risk']
  • TIME: ['48 hours']

TEXT PREVIEW:
[ARTICLE 26] 8. Deployers of high-risk AI systems that are public authorities, or Union institutions, bodies, offices or agencies shall comply with the registration obligations referred to in Article 49. When such deployers find that the high-risk AI system that they envisage using has not been regi

In [31]:
from typing import List, Optional

def retrieve_and_filter_chunks(
    query: str,
    hybrid_retriever,
    target_role: Optional[str] = None,
    target_risk: Optional[str] = None,
    fetch_k: int = 60,
    final_k: int = 5
) -> List:
    """
    Retrieves chunks using hybrid search and filters them by metadata.
    Fetches a large pool (fetch_k) to ensure enough chunks survive the filter.
    """
    for retriever in hybrid_retriever.retrievers:
        if hasattr(retriever, 'k'):
            retriever.k = fetch_k
        elif hasattr(retriever, 'search_kwargs'):
            retriever.search_kwargs["k"] = fetch_k

    raw_chunks = hybrid_retriever.invoke(query)

    filtered_chunks = []

    for chunk in raw_chunks:
        roles = [r.lower() for r in chunk.metadata.get("ner_LEGAL_ROLE", [])]
        risks = [r.lower() for r in chunk.metadata.get("ner_RISK_LEVEL", [])]

        role_match = True
        if target_role:
            role_match = any(target_role.lower() in r for r in roles)

        risk_match = True
        if target_risk:
            risk_match = any(target_risk.lower().replace("-", " ") in r.replace("-", " ") for r in risks)

        if role_match and risk_match:
            filtered_chunks.append(chunk)

    if len(filtered_chunks) < final_k:
        print(f"Warning: Only found {len(filtered_chunks)} exact metadata matches. Padding with top semantic matches.")
        for chunk in raw_chunks:
            if chunk not in filtered_chunks:
                filtered_chunks.append(chunk)
            if len(filtered_chunks) >= final_k:
                break

    return filtered_chunks[:final_k]

In [32]:
from langchain_core.prompts import PromptTemplate

def build_super_prompt(user_role: Optional[str], user_risk: Optional[str]) -> PromptTemplate:
    #  Prepare the strings first
    clean_risk = user_risk.replace('_', ' ').upper() if user_risk else "GENERAL"
    clean_role = user_role.upper() if user_role else "STAKEHOLDER"

    if user_role or user_risk:
        profile_desc = f"USER CATEGORY: {clean_risk}\nUSER ROLE: {clean_role}"
        mismatch_instruction = (
            f"STRICT HIERARCHY: Your primary filter is '{clean_risk}'. "
            f"1. Scan the context. Does it mention rules for '{clean_risk}'? "
            f"2. If the context ONLY mentions 'HIGH-RISK' and the user is '{clean_risk}', "
            "you MUST state that the obligations for High-Risk do NOT apply to them."
        )
    else:
        profile_desc = "USER CATEGORY: NOT SPECIFIED\nUSER ROLE: NOT SPECIFIED"
        mismatch_instruction = "Provide a general overview across all mentioned risk levels."

    template = f"""### SYSTEM ROLE:
You are an expert Legal Advisor for the EU AI Act. You provide precise, grounded, and safe legal analysis.

### USER PROFILE (DO NOT IGNORE):
{profile_desc}

### GENERATION PROTOCOL:
1. **LEGAL VERIFICATION STEP:** {mismatch_instruction}
2. **NO HALLUCINATION:** Use ONLY the provided 'Retrieved Legal Context'. If the information is not there, say: "The provided excerpts do not contain information applicable to your specific profile."
3. **STRUCTURE:** - Start with a direct answer regarding the user's specific status ({clean_risk}).
   - Provide a professional explanation with in-text citations like [Article 5] or [Recital 10].
   - If the context discusses a different risk level than the user's, explicitly explain the difference.
4. **SOURCES:** At the very end, add a horizontal line '---' followed by a bulleted list of the Articles/Recitals used.

### RETRIEVED LEGAL CONTEXT:
{{context}}

### USER QUESTION:
{{question}}

### FINAL ANALYSIS:
As a {clean_risk} {clean_role} under the AI Act, here is the analysis of your obligations:
"""
    return PromptTemplate(
        input_variables=["context", "question"],
        template=template
    )

In [ ]:
pip install langchain-huggingface accelerate

In [35]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('hf_gemma')
login(token=hf_token)


model_id = "google/gemma-2-2b-it"
print(f"Loading {model_id}...")

llm_tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# also try: temperature=0.01, repetition_penalty=1.2,do_sample=False
text_generation_pipeline = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    max_length=None,
    return_full_text=False
)

hf_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

def generate_final_answer(final_prompt: str) -> str:
    print("Generating answer... (this might take a few seconds)")
    response = hf_llm.invoke(final_prompt)
    return response.strip()

Loading google/gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'max_length', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
!pip install transformers datasets evaluate accelerate -U

In [45]:
import pandas as pd
import os

file_path = "ai_risk_dataset.csv.txt"

if os.path.exists(file_path):
    print(f"Loading dataset from: {file_path}")
    df = pd.read_csv(file_path)

    print("\n--- FIRST 5 ROWS ---")
    print(df.head())

    print("\n--- LABEL DISTRIBUTION ---")
    print(df['label'].value_counts())
else:
    print(f"Error: '{file_path}' not found.")

Loading dataset from: ai_risk_dataset.csv.txt

--- FIRST 5 ROWS ---
                                                text              label
0  An AI-powered social scoring system used by mu...  Unacceptable_Risk
1  A real-time biometric identification system de...  Unacceptable_Risk
2  An AI application designed to subconsciously m...  Unacceptable_Risk
3  A predictive policing tool used to identify an...  Unacceptable_Risk
4  A cognitive-behavioral manipulation AI used by...  Unacceptable_Risk

--- LABEL DISTRIBUTION ---
label
Unacceptable_Risk    75
High_Risk            75
Limited_Risk         75
Minimal_Risk         75
Name: count, dtype: int64


In [38]:
label_map = {"Minimal_Risk": 0, "Limited_Risk": 1, "High_Risk": 2, "Unacceptable_Risk": 3}
df['label'] = df['label'].map(label_map)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train = train_dataset.map(tokenize_function, batched=True) # batched=True process faster
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    probabilities = softmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    recall = recall_score(labels, predictions, average='macro')

    auc = roc_auc_score(labels, probabilities, multi_class='ovr', average='macro')

    # nir
    largest_class_count = max(np.bincount(labels))
    nir = largest_class_count / len(labels)

    return {
        "accuracy": accuracy,
        "f1_macro": f1,
        "recall_macro": recall,
        "roc_auc": auc,
        "NIR": nir
    }

training_args = TrainingArguments(
    output_dir="./ai_risk_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
print(f"Train: {len(train_df)} | Val: {len(eval_df)} | Test: {len(test_df)}")

Train: 240 | Val: 30 | Test: 30


In [40]:
import os

trainer.train()

print("\n--- TEST SET EVALUATION ---")
test_results = trainer.predict(tokenized_test)

if test_results.metrics:
    for key, value in test_results.metrics.items():
        print(f"  • {key}: {value:.4f}")

save_path = "distilBERT_classifier"


trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"\n Training completed. Architecture and tokenizer saved locally to: {save_path}")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Recall Macro,Roc Auc,Nir
1,1.292202,1.089174,0.800000,0.756932,0.785714,0.931351,0.266667
2,0.906178,0.700160,0.900000,0.896619,0.892857,0.984110,0.266667
3,0.555526,0.423581,0.966667,0.966063,0.964286,0.995606,0.266667
4,0.322383,0.301327,0.966667,0.966667,0.968750,1.000000,0.266667
5,0.201291,0.238664,0.966667,0.966667,0.968750,1.000000,0.266667
6,0.149385,0.208541,1.000000,1.000000,1.000000,1.000000,0.266667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



--- TEST SET EVALUATION ---


  • test_loss: 0.3941
  • test_accuracy: 0.8667
  • test_f1_macro: 0.8661
  • test_recall_macro: 0.8661
  • test_roc_auc: 0.9823
  • test_NIR: 0.2667
  • test_runtime: 0.6227
  • test_samples_per_second: 48.1750
  • test_steps_per_second: 6.4230


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 Training completed. Architecture and tokenizer saved locally to: distilBERT_classifier


In [41]:
log_history = trainer.state.log_history

metrics_per_epoch = {}

for entry in log_history:
    if "epoch" not in entry:
        continue

    epoch = round(entry["epoch"], 2)

    if epoch not in metrics_per_epoch:
        metrics_per_epoch[epoch] = {'Epoch': int(epoch)}

    if "loss" in entry and "eval_loss" not in entry:
        metrics_per_epoch[epoch]["Training Loss"] = entry["loss"]

    if "eval_loss" in entry:
        metrics_per_epoch[epoch]["Validation Loss"] = entry["eval_loss"]
        metrics_per_epoch[epoch]["Accuracy"] = entry.get("eval_accuracy")
        metrics_per_epoch[epoch]["F1 Macro"] = entry.get("eval_f1_macro")
        metrics_per_epoch[epoch]["Recall Macro"] = entry.get("eval_recall_macro")
        metrics_per_epoch[epoch]["Roc Auc"] = entry.get("eval_roc_auc")
        metrics_per_epoch[epoch]["Nir"] = entry.get("eval_NIR")

metrics_df = pd.DataFrame(list(metrics_per_epoch.values()))

metrics_df = metrics_df.dropna(subset=["Validation Loss"])

columns_order = [
    "Epoch", "Training Loss", "Validation Loss",
    "Accuracy", "F1 Macro", "Recall Macro", "Roc Auc", "Nir"
]
existing_columns = [col for col in columns_order if col in metrics_df.columns]
metrics_df = metrics_df[existing_columns]

print(metrics_df.to_string(index=False))

 Epoch  Training Loss  Validation Loss  Accuracy  F1 Macro  Recall Macro  Roc Auc      Nir
     1       1.292202         1.089174  0.800000  0.756932      0.785714 0.931351 0.266667
     2       0.906178         0.700160  0.900000  0.896619      0.892857 0.984110 0.266667
     3       0.555526         0.423581  0.966667  0.966063      0.964286 0.995606 0.266667
     4       0.322383         0.301327  0.966667  0.966667      0.968750 1.000000 0.266667
     5       0.201291         0.238664  0.966667  0.966667      0.968750 1.000000 0.266667
     6       0.149385         0.208541  1.000000  1.000000      1.000000 1.000000 0.266667


In [42]:
log_history = trainer.state.log_history
data = []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        data.append({
            'Epoch': entry["epoch"],
            'Loss': entry["loss"],
            'Dataset': 'Training Loss'
        })


    if "eval_loss" in entry:
        data.append({
            'Epoch': entry["epoch"],
            'Loss': entry["eval_loss"],
            'Dataset': 'Validation Loss'
        })

df_base = pd.DataFrame(data)

if df_base[df_base['Dataset'] == 'Training Loss'].empty:
    print(" WARNING: No Training Loss data found in logs.")


all_frames = sorted(df_base['Epoch'].unique())
animated_data = []

for current_frame in all_frames:
    df_snapshot = df_base[df_base['Epoch'] <= current_frame].copy()
    df_snapshot['Frame'] = current_frame
    animated_data.append(df_snapshot)

df_animated = pd.concat(animated_data, ignore_index=True)


fig = px.line(
    df_animated,
    x="Epoch",
    y="Loss",
    color="Dataset",
    animation_frame="Frame",
    animation_group="Dataset",
    markers=True,
    title="Evolution of Training vs. Validation Loss",
    template="plotly_white",
    color_discrete_map={"Training Loss": "blue", "Validation Loss": "orange"},
    range_x=[0, df_base['Epoch'].max() * 1.05],
    range_y=[0, df_base['Loss'].max() * 1.1]
)

fig.show()

In [43]:
print(f"Accuracy: {test_results.metrics['test_accuracy'] * 100:.2f}%")
print(f"F1 Score: {test_results.metrics['test_f1_macro']:.4f}")
print(f"Recall:   {test_results.metrics['test_recall_macro']:.4f}")
print(f"ROC AUC:  {test_results.metrics['test_roc_auc']:.4f}")
print(f"NIR:      {test_results.metrics['test_NIR'] * 100:.2f}%")

Accuracy: 86.67%
F1 Score: 0.8661
Recall:   0.8661
ROC AUC:  0.9823
NIR:      26.67%


In [ ]:
import pandas as pd
import os

file_path = "ai_risk_dataset.csv.txt"

if os.path.exists(file_path):
    print(f"Loading dataset from: {file_path}")
    df = pd.read_csv(file_path)

    print("\n--- FIRST 5 ROWS ---")
    print(df.head())

    print("\n--- LABEL DISTRIBUTION ---")
    print(df['label'].value_counts())
else:
    print(f"Error: '{file_path}' not found.")

In [46]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from scipy.special import softmax

label_map = {"Minimal_Risk": 0, "Limited_Risk": 1, "High_Risk": 2, "Unacceptable_Risk": 3}
df['label'] = df['label'].map(label_map)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)
test_dataset = Dataset.from_pandas(test_df)


model_checkpoint = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=4)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probabilities = softmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    recall = recall_score(labels, predictions, average='macro')
    auc = roc_auc_score(labels, probabilities, multi_class='ovr', average='macro')

    largest_class_count = max(np.bincount(labels))
    nir = largest_class_count / len(labels)

    return {
        "accuracy": accuracy,
        "f1_macro": f1,
        "recall_macro": recall,
        "roc_auc": auc,
        "NIR": nir
    }


training_args = TrainingArguments(
    output_dir="./legal_ai_risk_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [47]:
import os

trainer.train()

print("\n--- TEST SET EVALUATION ---")
test_results = trainer.predict(tokenized_test)

if test_results.metrics:
    for key, value in test_results.metrics.items():
        print(f"  • {key}: {value:.4f}")

save_path = "legalBERT_classifier"


trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"\n Training completed. Architecture and tokenizer saved locally to: {save_path}")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Recall Macro,Roc Auc,Nir
1,1.370815,1.300663,0.400000,0.277174,0.406250,0.784347,0.266667
2,1.240285,1.002902,0.733333,0.725757,0.736607,0.955304,0.266667
3,0.912061,0.725229,0.800000,0.781944,0.794643,0.966006,0.266667
4,0.584639,0.592700,0.833333,0.809926,0.825893,0.972976,0.266667
5,0.438646,0.475157,0.933333,0.935417,0.937500,0.978429,0.266667
6,0.371792,0.464994,0.866667,0.856124,0.861607,0.982293,0.266667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


--- TEST SET EVALUATION ---


  • test_loss: 0.4634
  • test_accuracy: 0.8667
  • test_f1_macro: 0.8609
  • test_recall_macro: 0.8661
  • test_roc_auc: 0.9744
  • test_NIR: 0.2667
  • test_runtime: 1.0788
  • test_samples_per_second: 27.8090
  • test_steps_per_second: 3.7080


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 Training completed. Architecture and tokenizer saved locally to: legalBERT_classifier


In [48]:
log_history = trainer.state.log_history

metrics_per_epoch = {}

for entry in log_history:
    if "epoch" not in entry:
        continue

    epoch = round(entry["epoch"], 2)

    if epoch not in metrics_per_epoch:
        metrics_per_epoch[epoch] = {'Epoch': int(epoch)}

    if "loss" in entry and "eval_loss" not in entry:
        metrics_per_epoch[epoch]["Training Loss"] = entry["loss"]

    if "eval_loss" in entry:
        metrics_per_epoch[epoch]["Validation Loss"] = entry["eval_loss"]
        metrics_per_epoch[epoch]["Accuracy"] = entry.get("eval_accuracy")
        metrics_per_epoch[epoch]["F1 Macro"] = entry.get("eval_f1_macro")
        metrics_per_epoch[epoch]["Recall Macro"] = entry.get("eval_recall_macro")
        metrics_per_epoch[epoch]["Roc Auc"] = entry.get("eval_roc_auc")
        metrics_per_epoch[epoch]["Nir"] = entry.get("eval_NIR")

metrics_df = pd.DataFrame(list(metrics_per_epoch.values()))

metrics_df = metrics_df.dropna(subset=["Validation Loss"])

columns_order = [
    "Epoch", "Training Loss", "Validation Loss",
    "Accuracy", "F1 Macro", "Recall Macro", "Roc Auc", "Nir"
]
existing_columns = [col for col in columns_order if col in metrics_df.columns]
metrics_df = metrics_df[existing_columns]

print(metrics_df.to_string(index=False))

 Epoch  Training Loss  Validation Loss  Accuracy  F1 Macro  Recall Macro  Roc Auc      Nir
     1       1.370815         1.300663  0.400000  0.277174      0.406250 0.784347 0.266667
     2       1.240285         1.002902  0.733333  0.725757      0.736607 0.955304 0.266667
     3       0.912061         0.725229  0.800000  0.781944      0.794643 0.966006 0.266667
     4       0.584639         0.592700  0.833333  0.809926      0.825893 0.972976 0.266667
     5       0.438646         0.475157  0.933333  0.935417      0.937500 0.978429 0.266667
     6       0.371792         0.464994  0.866667  0.856124      0.861607 0.982293 0.266667


In [49]:
log_history = trainer.state.log_history
data = []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        data.append({
            'Epoch': entry["epoch"],
            'Loss': entry["loss"],
            'Dataset': 'Training Loss'
        })


    if "eval_loss" in entry:
        data.append({
            'Epoch': entry["epoch"],
            'Loss': entry["eval_loss"],
            'Dataset': 'Validation Loss'
        })

df_base = pd.DataFrame(data)

if df_base[df_base['Dataset'] == 'Training Loss'].empty:
    print(" WARNING: No Training Loss data found in logs.")


all_frames = sorted(df_base['Epoch'].unique())
animated_data = []

for current_frame in all_frames:
    df_snapshot = df_base[df_base['Epoch'] <= current_frame].copy()
    df_snapshot['Frame'] = current_frame
    animated_data.append(df_snapshot)

df_animated = pd.concat(animated_data, ignore_index=True)


fig = px.line(
    df_animated,
    x="Epoch",
    y="Loss",
    color="Dataset",
    animation_frame="Frame",
    animation_group="Dataset",
    markers=True,
    title="Evolution of Training vs. Validation Loss",
    template="plotly_white",
    color_discrete_map={"Training Loss": "blue", "Validation Loss": "orange"},
    range_x=[0, df_base['Epoch'].max() * 1.05],
    range_y=[0, df_base['Loss'].max() * 1.1]
)

fig.show()

In [50]:
print(f"Accuracy: {test_results.metrics['test_accuracy'] * 100:.2f}%")
print(f"F1 Score: {test_results.metrics['test_f1_macro']:.4f}")
print(f"Recall:   {test_results.metrics['test_recall_macro']:.4f}")
print(f"ROC AUC:  {test_results.metrics['test_roc_auc']:.4f}")
print(f"NIR:      {test_results.metrics['test_NIR'] * 100:.2f}%")

Accuracy: 86.67%
F1 Score: 0.8609
Recall:   0.8661
ROC AUC:  0.9744
NIR:      26.67%


In [51]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2026-03-19 18:06:45--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-03-19 18:06:46--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-03-19 18:06:47--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [53]:
import pandas as pd
import os

file_path = "ai_risk_dataset.csv.txt"

if os.path.exists(file_path):
    print(f"Loading dataset from: {file_path}")
    df = pd.read_csv(file_path)
else:
    print(f"Error: '{file_path}' not found.")

Loading dataset from: ai_risk_dataset.csv.txt


In [54]:
df = pd.read_csv(file_path)
df['text'] = df['text'].str.replace('-', ' ', regex=False)

label_map = {"Minimal_Risk": 0, "Limited_Risk": 1, "High_Risk": 2, "Unacceptable_Risk": 3}
df['label'] = df['label'].map(label_map)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])


X_train, y_train = train_df['text'].to_numpy(), train_df['label'].to_numpy()
X_eval, y_eval = eval_df['text'].to_numpy(), eval_df['label'].to_numpy()
X_test, y_test = test_df['text'].to_numpy(), test_df['label'].to_numpy()

In [55]:
embeddings_index = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32') # embddings
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} pre-trained word vectors.")

Found 400000 pre-trained word vectors.


In [56]:
from tensorflow.keras.layers import TextVectorization
max_features = 1500
sequence_length = 100

vectorizer = TextVectorization(
    max_tokens=max_features,
    output_sequence_length=sequence_length
)

vectorizer.adapt(X_train)

voc = vectorizer.get_vocabulary()
word_index = dict(zip(voc, range(len(voc))))
num_tokens = len(voc)
embedding_dim = 100 #  match the GloVe file downloaded (100d)

embedding_matrix = np.zeros((num_tokens, embedding_dim))

hits = 0
misses = 0
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
        hits += 1
    else:
        misses += 1

print(f"Converted {hits} words ({misses} misses)")

Converted 1486 words (8 misses)


In [57]:
missing_words = []

for word in word_index.keys():
    if word not in embeddings_index:
        missing_words.append(word)

print("missing words:", missing_words)

missing words: ['', '[UNK]', np.str_('chatbot'), np.str_('deepfake'), np.str_('neurodivergent'), np.str_('librarys'), np.str_('gamified'), np.str_('deprioritizes')]


In [58]:
from tensorflow.keras.layers import SpatialDropout1D
model = Sequential([
    Input(shape=(1,), dtype=tf.string),
    vectorizer,

    Embedding(
        input_dim=num_tokens,
        output_dim=embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        trainable=False,
    ),

    SpatialDropout1D(0.2),

    Bidirectional(
        GRU(
            32,
            dropout=0.2,
            recurrent_dropout=0.1,
            return_sequences=False
        )
    ),

    Dense(32, activation='relu'),
    Dropout(0.3),

    Dense(4, activation='softmax')
])


total_steps = (len(X_train) // 8) * 10 # epochs = 10

# linear decay
lr_scheduler = PolynomialDecay(
    initial_learning_rate=1e-3,
    end_learning_rate=1e-5,
    decay_steps=total_steps,
    power=1.0
)


model.compile(
    optimizer=AdamW(learning_rate=lr_scheduler, weight_decay=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [59]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_eval, y_eval),
    epochs=10,
    batch_size=8,
    callbacks=callbacks
)

Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 29s 590ms/step - accuracy: 0.2458 - loss: 1.4145 - val_accuracy: 0.3000 - val_loss: 1.3656
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 26s 880ms/step - accuracy: 0.2833 - loss: 1.3729 - val_accuracy: 0.2333 - val_loss: 1.3695
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 32s 554ms/step - accuracy: 0.3500 - loss: 1.3561 - val_accuracy: 0.3333 - val_loss: 1.3561
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 22s 612ms/step - accuracy: 0.3292 - loss: 1.3695 - val_accuracy: 0.4333 - val_loss: 1.3381
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 550ms/step - accuracy: 0.3375 - loss: 1.3491 - val_accuracy: 0.5000 - val_loss: 1.3270
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 560ms/step - accuracy: 0.4167 - loss: 1.3190 - val_accuracy: 0.5000 - val_loss: 1.3118
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 562ms/step - accuracy: 0.4083 - loss: 1.3225 - val_accuracy: 0.5000 - val_loss: 1.2950
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 16s 549ms/step - accuracy: 0.4000 - loss: 1.3116 - val_accu

In [60]:
metrics_dict = history.history
metrics_df = pd.DataFrame(metrics_dict)
metrics_df.insert(0, 'Epoch', range(1, len(metrics_df) + 1))
print(metrics_df.to_string(index=False))

 Epoch  accuracy     loss  val_accuracy  val_loss
     1  0.245833 1.414531      0.300000  1.365587
     2  0.283333 1.372905      0.233333  1.369488
     3  0.350000 1.356068      0.333333  1.356077
     4  0.329167 1.369532      0.433333  1.338081
     5  0.337500 1.349125      0.500000  1.327014
     6  0.416667 1.319040      0.500000  1.311798
     7  0.408333 1.322547      0.500000  1.295026
     8  0.400000 1.311554      0.433333  1.278629
     9  0.475000 1.267474      0.433333  1.270331
    10  0.462500 1.292163      0.466667  1.266999


In [61]:
# The model.fit() process in Keras returns a history object.
# extract the training and validation losses from it:
train_loss = history.history['loss']
eval_loss = history.history['val_loss']

# Keras epochs are 0-indexed
epochs = [e + 1 for e in history.epoch]

data = []

for e, l in zip(epochs, train_loss):
    data.append({'Epoch': e, 'Loss': l, 'Dataset': 'Training Loss'})
for e, l in zip(epochs, eval_loss):
    data.append({'Epoch': e, 'Loss': l, 'Dataset': 'Validation Loss'})

df_base = pd.DataFrame(data)

animated_data = []
unique_epochs = sorted(list(set(epochs)))


for current_frame in unique_epochs:
    df_snapshot = df_base[df_base['Epoch'] <= current_frame].copy()
    df_snapshot['Frame'] = current_frame
    animated_data.append(df_snapshot)

df_animated = pd.concat(animated_data, ignore_index=True)

fig = px.line(
    df_animated,
    x="Epoch",
    y="Loss",
    color="Dataset",
    animation_frame="Frame",
    animation_group="Dataset",
    markers=True,
    range_x=[0, df_base['Epoch'].max() + 0.2],
    range_y=[df_base['Loss'].min() * 0.9, df_base['Loss'].max() * 1.1],
    title="Evolution of Training vs. Validation Loss (GRU Model)",
    template="plotly_white",
    color_discrete_map={"Training Loss": "blue", "Validation Loss": "orange"}
)

fig.show()

In [ ]:
probabilities = model.predict(X_test)
predictions = np.argmax(probabilities, axis=-1)

accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average='macro')
recall = recall_score(y_test, predictions, average='macro')
auc = roc_auc_score(y_test, probabilities, multi_class='ovr', average='macro')

largest_class_count = max(np.bincount(y_test))
nir = largest_class_count / len(y_test)

results = {
    "accuracy": accuracy,
    "f1_macro": f1,
    "recall_macro": recall,
    "roc_auc": auc,
    "NIR": nir
}

In [63]:
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

accuracy: 0.7333
f1_macro: 0.7345
recall_macro: 0.7277
roc_auc: 0.9250
NIR: 0.2667
